# LLaVA-1.5-7B QLoRA Fine-Tuning for Physics Diagrams

This notebook fine-tunes LLaVA-1.5-7B using QLoRA (4-bit quantization + LoRA) on physics diagram QA data.
Designed for **Kaggle T4 GPUs** (16GB VRAM).

## Setup
- Runtime: GPU T4 x2
- Estimated training time: 8-12 hours
- Dataset: ScienceQA (physics) + synthetic GPT-4o pairs

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 \
    accelerate>=0.28.0 datasets>=2.18.0 Pillow>=10.2.0 sentencepiece

In [ ]:
import json
import torch
from pathlib import Path
from PIL import Image
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    LlavaForConditionalGeneration,
    TrainingArguments,
    Trainer,
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Configuration

In [ ]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"
OUTPUT_DIR = "/kaggle/working/llava_physics_qlora"

# LoRA hyperparameters
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# Training hyperparameters
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_LENGTH = 1024

## Load Training Data

Upload your `train.json` and `val.json` from `data/processed/finetune/` to the Kaggle dataset,
or use the cells below to download and prepare directly.

In [ ]:
# Option 1: Load from uploaded Kaggle dataset
DATA_DIR = Path("/kaggle/input/physics-finetune-data")

# Option 2: If running locally, point to your processed data
# DATA_DIR = Path("../data/processed/finetune")

with open(DATA_DIR / "train.json") as f:
    train_data = json.load(f)
with open(DATA_DIR / "val.json") as f:
    val_data = json.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Val samples: {len(val_data)}")
print(f"\nSample entry keys: {train_data[0].keys() if train_data else 'empty'}")

## Load Model with 4-bit Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"Model loaded. Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Apply LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

## Data Collator

In [ ]:
class LlavaDataCollator:
    """Custom data collator for LLaVA fine-tuning with images."""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts = []
        images = []

        for ex in examples:
            convs = ex["conversations"]
            human_msg = convs[0]["value"]
            gpt_msg = convs[1]["value"]
            text = f"USER: {human_msg}\nASSISTANT: {gpt_msg}"
            texts.append(text)

            img_path = ex.get("image", "")
            if img_path and Path(img_path).exists():
                img = Image.open(img_path).convert("RGB")
                images.append(img)
            else:
                images.append(Image.new("RGB", (224, 224)))

        batch = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        labels = batch["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch["labels"] = labels
        return batch

data_collator = LlavaDataCollator(processor)

## Training

In [ ]:
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

## Save Adapter

In [ ]:
adapter_dir = f"{OUTPUT_DIR}/final_adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}")

# Verify
import os
adapter_files = os.listdir(adapter_dir)
print(f"Files: {adapter_files}")
total_size = sum(os.path.getsize(os.path.join(adapter_dir, f)) for f in adapter_files)
print(f"Total adapter size: {total_size / 1e6:.1f} MB")

## Quick Inference Test

In [ ]:
test_question = "A block of mass 5 kg is on a frictionless inclined plane at 30 degrees. What is the acceleration of the block down the incline?"

prompt = f"USER: {test_question}\nASSISTANT:"
inputs = processor(text=prompt, return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)

response = processor.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {test_question}")
print(f"A: {response}")